# nuScenes → Google Drive — CPU-only archive download
Notebook revision **NUSCENES-DRIVE-DOWNLOAD-2026-10-28-r1**.

This notebook downloads all **850 trainval scenes' annotated keyframes**, metadata and lidarseg labels directly from the verified public official endpoints into your Google Drive. No dataset download to your Mac, old M63/M67 runtime, CUDA extension, model inference, training or phone update is involved.

**Runtime → Change runtime type → Hardware accelerator: None.** Stay on CPU throughout.

Default: all ten `_keyframes.tgz` archives + metadata + lidarseg: **45.53 GB compressed**. Optional `VARIANT='full'`: all ten `_blobs.tgz` archives + metadata + lidarseg: **315.05 GB compressed**. Do not choose full sensor sweeps merely to obtain all scenes; all scenes' keyframes are already included in the default.

Archive storage is separate from prepared training shards. The preflight reserves another 20 GB, but this is not a promise that later extracted/prepared data fit. We will process archives in bounded Colab-local batches in a separate coverage-preparation workflow.

**Accuracy policy:** no ±10 cm pass/fail threshold. Future coverage includes 0–1, 1–2, 2–4 and 4–6 m. Downloading all scenes does not establish adequate sub-metre truth.


## 1. Mount Drive and load the published project main branch

Authorize Drive mounting yourself. No nuScenes password or signed links are needed for the configured public endpoints. Do not run another downloader against the same dataset folder from another notebook/runtime.

Run from the top after a Colab reset. Existing complete archives and verified partial chunks are reused; unrelated files are never overwritten or deleted.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import datetime, json, os, shlex, subprocess, sys

NOTEBOOK_REVISION = 'NUSCENES-DRIVE-DOWNLOAD-2026-10-28-r1'
MOBILE = Path('/content/mobile_adas3d')
DATASET = Path('/content/drive/MyDrive/mobile_adas3d_datasets/nuscenes_v1_0')
LOGS = DATASET / 'colab_logs'
if os.environ.get('COLAB_GPU') not in (None, '', '0'):
    raise RuntimeError('This is CPU-only. Set Hardware accelerator to None, reconnect, and rerun from the top.')

def run_logged(command, name, cwd=None):
    command = [str(x) for x in command]
    LOGS.mkdir(parents=True, exist_ok=True)
    stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    log = LOGS / (name + '_' + stamp + '.log')
    print('+', shlex.join(command), '\nDurable log:', log, flush=True)
    tail = []
    with log.open('w') as output:
        process = subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                output.write(line); output.flush()
                tail = (tail + [line.rstrip()])[-25:]
        finally:
            process.stdout.close()
        status = process.wait()
        output.write('\nEXIT CODE: ' + str(status) + '\n')
    if status:
        raise RuntimeError(f'Exit {status}; preserve all files and inspect {log}\n' + '\n'.join(tail))
    return log

repo_url = '/'.join(['https:', '', 'github.com', 'Ali-RT', 'mobile_adas3d.git'])
if not MOBILE.exists():
    run_logged(['git', 'clone', repo_url, MOBILE], 'clone', '/content')
branch = subprocess.check_output(['git', 'branch', '--show-current'], cwd=MOBILE, text=True).strip()
if branch != 'main':
    raise RuntimeError(f'Use main, not {branch!r}; do not reset or delete the existing checkout.')
run_logged(['git', 'pull', '--ff-only'], 'update_main', MOBILE)
SCRIPT = MOBILE / 'scripts/download_nuscenes_to_drive.py'
CONFIG = MOBILE / 'configs/nuscenes_drive_sources_v1.json'
for required in [SCRIPT, CONFIG, MOBILE/'scripts/parking_edge_common.py',
                 MOBILE/'tests/test_nuscenes_drive_download.py']:
    if not required.is_file():
        raise FileNotFoundError(f'{required} is missing. Use the published main-branch notebook; do not start downloading.')
print(NOTEBOOK_REVISION, '\nDrive dataset folder:', DATASET)


## 2. Choose archive type and run the downloader tests

Leave `VARIANT='keyframes'` and all ten parts selected for the current benchmark. The full-sensor option is available but is roughly seven times larger. Existing archives from a different variant are preserved, not removed.

Only the Python standard library is needed. Tests use a tiny local HTTP server, not the real dataset.


In [ ]:
VARIANT = 'keyframes'  #@param ['keyframes', 'full']
PARTS = list(range(1, 11))
RESERVE_GB = 20.0
MANUAL_FREE_DRIVE_GB = 0.0  # Set actual free quota from drive.google.com/drive/quota ONLY if automatic quota lookup fails.
PREFLIGHT_READY = False
DOWNLOAD_READY = False

if VARIANT not in ('keyframes', 'full') or sorted(set(PARTS)) != sorted(PARTS) or not set(PARTS) <= set(range(1, 11)) or not PARTS:
    raise ValueError('Choose keyframes/full and distinct parts 1-10.')
run_logged([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests',
            '-p', 'test_nuscenes_drive_download.py', '-v'], 'downloader_tests', MOBILE)
def command(stage, free_gb=None):
    values = [sys.executable, '-u', SCRIPT, stage, '--config', CONFIG,
              '--root', DATASET, '--variant', VARIANT, '--reserve-gb', str(RESERVE_GB)]
    if PARTS != list(range(1, 11)):
        values += ['--parts', *[str(i) for i in PARTS]]
    if free_gb is not None:
        values += ['--available-drive-gb', str(free_gb)]
    return values
TOKEN = 'all' if PARTS == list(range(1, 11)) else '_'.join(str(i) for i in PARTS)
MANIFEST = DATASET / f'download_manifest_{VARIANT}_{TOKEN}.json'
print('Selected:', VARIANT, PARTS)


## 3. Check real Drive quota, URLs and download budget — no data downloaded yet

The Drive mount's filesystem free-space number is **not** an account quota check. This cell reads the account's quota via the Drive API after your authorization. If the account/API cannot report a finite limit, enter the actual free GB in `MANUAL_FREE_DRIVE_GB` in section 2 and rerun sections 2–3. Do not guess.

The plan subtracts existing owned archive bytes and checks the **remaining** download plus reserve. Existing complete files will still be SHA256-checked by the download stage. Colab temporary disk and Drive quota are separate.


In [ ]:
FREE_DRIVE_GB = None
try:
    from google.colab import auth
    auth.authenticate_user()
    import google.auth
    from google.auth.transport.requests import AuthorizedSession
    credentials, _ = google.auth.default()
    quota_response = AuthorizedSession(credentials).get(
        'https://www.googleapis.com/drive/v3/about',
        params={'fields': 'storageQuota'}, timeout=45)
    if quota_response.status_code == 200:
        quota = quota_response.json()['storageQuota']
        if quota.get('limit'):
            remaining = max(0, int(quota['limit']) - int(quota.get('usage', 0)))
            FREE_DRIVE_GB = remaining / 1e9
    else:
        print('Drive quota API returned status', quota_response.status_code)
except Exception as error:
    print('Automatic Drive quota lookup unavailable:', type(error).__name__)

if FREE_DRIVE_GB is None:
    if MANUAL_FREE_DRIVE_GB <= 0:
        raise RuntimeError('Enter actual free Google Drive GB in MANUAL_FREE_DRIVE_GB in section 2, then rerun sections 2 and 3. No dataset download started.')
    FREE_DRIVE_GB = float(MANUAL_FREE_DRIVE_GB)
    print('Using user-reported free Drive quota.')
print(f'Free Drive quota: {FREE_DRIVE_GB:.2f} GB')
run_logged(command('plan', FREE_DRIVE_GB), 'archive_preflight', MOBILE)
PREFLIGHT_READY = True
print('Preflight passed. Review the archive list and storage budget, then explicitly enable the next cell.')


## 4. Download/resume the selected archives — CPU only

After reviewing the plan, switch **CONFIRM_DOWNLOAD** on below. Keep just one runtime writing this folder. For an interruption, reconnect and rerun from section 1 with the same folder/variant/parts. Do not delete partials, receipts or state.

The helper:
- verifies source Content-Length and observed ETag before transfer;
- re-hashes committed 64 MiB chunks before resuming;
- refuses a server that ignores or misreports a resume range;
- preserves conflicting/unowned files;
- checks final read-back SHA256 and gzip/bzip2 CRC before marking complete.

**Compression checks can take time and use CPU, not GPU.** They stream decompressed bytes without saving an extracted dataset. Multipart ETags are not treated as plain-file MD5 checksums. Most new archives do not have a publisher-signed SHA256; the completion receipt records the locally observed digest.


In [ ]:
CONFIRM_DOWNLOAD = False  #@param {type:"boolean"}
if not PREFLIGHT_READY:
    raise RuntimeError('Run sections 1-3 first.')
if not CONFIRM_DOWNLOAD:
    raise RuntimeError('No download started. After reviewing section 3, enable CONFIRM_DOWNLOAD and rerun this cell.')
run_logged(command('download', FREE_DRIVE_GB), 'archive_download', MOBILE)
DOWNLOAD_READY = True
print('All requested archives downloaded and individually verified. No extraction or training occurred.')


## 5. Inspect the durable result and optional deep recheck

The download stage already performs read-back hashes and compression CRC. The optional recheck repeats those expensive reads; it is not needed after every session.

The archives and per-file receipts stay in Drive. **This is not yet a prepared 0–6 m training dataset.** Next, we will audit additional scenes and prepare bounded, versioned input shards on CPU, preserving frozen recording-log roles and already-used evaluation IDs. Do not allocate a GPU or start an old training notebook merely because download completed.


In [ ]:
if not MANIFEST.is_file():
    raise FileNotFoundError(f'{MANIFEST} is not ready. Complete/resume section 4 first.')
result = json.loads(MANIFEST.read_text())
if not result['complete'] or result['variant'] != VARIANT or result['parts'] != PARTS:
    raise RuntimeError('Download manifest does not match this request.')
print(json.dumps({
    'complete': result['complete'],
    'archives': len(result['archives']),
    'compressed_GB': sum(r['size_bytes'] for r in result['archives']) / 1e9,
    'all_compression_CRC_checked': all(r['compression_crc_validated'] for r in result['archives']),
    'training_performed': result['training_performed'],
    'benchmark_data_prepared': result['benchmark_data_prepared'],
    'manifest': str(MANIFEST)
}, indent=2))
DEEP_RECHECK = False  #@param {type:"boolean"}
if DEEP_RECHECK:
    run_logged(command('verify'), 'archive_deep_recheck', MOBILE)
print('Return the small download_manifest JSON for review; do not download the dataset back to the Mac.')
